# 1. Exploratory Data Analysis

**Objetivo**: Explorar visual y estadísticamente el dataset de relojes para informar las decisiones de preprocesamiento, augmentación y arquitectura del modelo.

**Dataset**: 2,553 imágenes de relojes de 70 marcas con precios de $31 a $3,675.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

from src.utils import load_config
from src.utils.visualization import set_style, plot_price_distribution, plot_brand_analysis
from src.data import prepare_metadata, create_splits, get_transforms, extract_text_features, TEXT_FEATURES

set_style()
config = load_config('../configs/base.yaml')
config['data']['metadata_path'] = '../data/metadata.csv'
config['data']['root_dir'] = '../data/raw'
img_dir = Path(config['data']['root_dir'])

## 1.1 Visión general del dataset

In [ ]:
df = prepare_metadata(config)
print(f'Total imágenes: {len(df)}')
print(f'Marcas únicas: {df["brand"].nunique()}')
print(f'Rango de precios: ${df["price_clean"].min():.0f} – ${df["price_clean"].max():.0f}')
print(f'Precio medio: ${df["price_clean"].mean():.0f} | Mediana: ${df["price_clean"].median():.0f}')
print(f'\nColumnas: {df.columns.tolist()}')
print(f'Valores nulos: {df.isnull().sum().sum()}')
df.head(10)

## 1.2 Distribución de precios

La distribución de precios está fuertemente sesgada a la derecha (skew positivo). La transformación log1p estabiliza la varianza y convierte la distribución en algo más cercano a una gaussiana, facilitando el entrenamiento de la red.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['price_clean'], bins=50, edgecolor='white', alpha=0.8, color='steelblue')
axes[0].set_title('Distribución de precios (escala original)')
axes[0].set_xlabel('Precio ($)')
axes[0].axvline(df['price_clean'].median(), color='red', linestyle='--', label=f'Mediana: ${df["price_clean"].median():.0f}')
axes[0].legend()

axes[1].hist(np.log1p(df['price_clean']), bins=50, edgecolor='white', alpha=0.8, color='coral')
axes[1].set_title('Distribución de precios (log1p)')
axes[1].set_xlabel('log(1 + Precio)')
axes[1].axvline(np.log1p(df['price_clean'].median()), color='red', linestyle='--', label=f'Mediana log')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/price_distribution.png', dpi=150)
print(f'Skewness original: {df["price_clean"].skew():.2f}')
print(f'Skewness log1p: {np.log1p(df["price_clean"]).skew():.2f}')

**Decisión**: Aplicar transformación log1p al target. El skewness se reduce drásticamente, lo que permite al modelo aprender de forma más equilibrada en todos los rangos de precio.

## 1.3 Distribución por rangos de precio

In [ ]:
bins = [0, 100, 250, 500, 1000, 5000]
labels = ['<$100', '$100-250', '$250-500', '$500-1K', '$1K+']
df['price_range'] = pd.cut(df['price_clean'], bins=bins, labels=labels)
counts = df['price_range'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
counts.plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Distribución por rango de precio')
ax.set_ylabel('Nº imágenes')
for i, v in enumerate(counts):
    ax.text(i, v + 15, f'{v} ({v/len(df)*100:.0f}%)', ha='center', fontsize=9)
plt.xticks(rotation=0)
plt.tight_layout()

El 68% de los relojes están en el rango $100-$500. Los relojes de más de $1,000 representan solo el 9% del dataset, lo que hace más difícil predecir precios altos con precisión.

## 1.4 Análisis de marcas

In [ ]:
plot_brand_analysis(df)
plt.savefig('../outputs/figures/brand_analysis.png', dpi=150)

Las marcas más representadas (Tissot, Daniel Wellington, Nixon) están en el rango medio. Las marcas caras (Montblanc, Versace, Salvatore Ferragamo) tienen pocas muestras pero precios altos. Esta información de marca será clave como feature adicional en el modelo (V3+).

## 1.5 Propiedades de las imágenes

In [ ]:
sizes = [Image.open(img_dir / fname).size for fname in df['image_name']]
widths, heights = zip(*sizes)
print(f'Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')
print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')
print(f'Aspect ratio único: {set(round(w/h, 2) for w, h in sizes)}')
print(f'\nTodas las imágenes son {widths[0]}×{heights[0]} — no hay variabilidad de tamaño.')

**Hallazgo**: Todas las imágenes son exactamente 406×512 píxeles. Esto simplifica el preprocesamiento ya que no necesitamos gestionar aspect ratios variables. Usaremos resize a 224×224 para el modelo.

## 1.6 Muestras visuales por rango de precio

In [ ]:
price_bins = {
    'Budget (<$100)': df[df['price_clean'] < 100],
    'Mid ($100-500)': df[(df['price_clean'] >= 100) & (df['price_clean'] < 500)],
    'Premium ($500-1K)': df[(df['price_clean'] >= 500) & (df['price_clean'] < 1000)],
    'Luxury ($1K+)': df[df['price_clean'] >= 1000]
}

fig, axes = plt.subplots(4, 5, figsize=(16, 14))
for row, (label, subset) in enumerate(price_bins.items()):
    samples = subset.sample(min(5, len(subset)), random_state=42)
    for col, (_, s) in enumerate(samples.iterrows()):
        img = Image.open(img_dir / s['image_name'])
        axes[row, col].imshow(img)
        axes[row, col].set_title(f'${s["price_clean"]:.0f}', fontsize=9)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(label, fontsize=11, rotation=0, ha='right', va='center')
plt.suptitle('Muestras por rango de precio', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/figures/sample_images_by_price.png', dpi=150)

**Observaciones visuales clave**:
- **Budget**: Relojes digitales, plástico, fitness bands, diseños simples
- **Mid**: Correas de cuero, cajas más trabajadas, marcas visibles
- **Premium**: Brazaletes metálicos, chronographs, esferas complejas
- **Luxury**: Acero/oro, acabados premium, detalles de alta gama

El modelo debería poder captar estas diferencias visuales (material, complejidad de esfera, tipo de correa).

## 1.7 Features textuales del nombre

El campo `name` contiene información muy relevante: tipo de mecanismo, materiales, estilo. Extraemos 21 features binarias agrupadas en categorías.

In [ ]:
# Extraer features textuales para todo el dataset
text_feats = np.stack([extract_text_features(name) for name in df['name']])
feat_names = list(TEXT_FEATURES.keys())

# Frecuencia y precio medio por feature
rows = []
for i, fname in enumerate(feat_names):
    mask = text_feats[:, i] == 1
    if mask.sum() > 0:
        rows.append({
            'Feature': fname,
            'Count': int(mask.sum()),
            'Avg Price': f'${df[mask]["price_clean"].mean():.0f}',
            'Median Price': f'${df[mask]["price_clean"].median():.0f}',
        })
feat_df = pd.DataFrame(rows).sort_values('Count', ascending=False)
print(f'Features extraídas: {len(feat_names)}')
print(f'Muestras con al menos 1 feature: {(text_feats.sum(axis=1) > 0).sum()} / {len(df)}')
feat_df

Las features textuales tienen una correlación clara con el precio: "automatic" (mediana $950) vs "smartwatch" (mediana $139) vs "silicone" (mediana $69). Estas features complementan la información visual de la CNN.

## 1.8 Splits train / val / test

In [ ]:
train_df, val_df, test_df = create_splits(df, config)
print(f'Train: {len(train_df)} ({len(train_df)/len(df)*100:.0f}%)')
print(f'Val:   {len(val_df)} ({len(val_df)/len(df)*100:.0f}%)')
print(f'Test:  {len(test_df)} ({len(test_df)/len(df)*100:.0f}%)')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, split) in zip(axes, [('Train', train_df), ('Val', val_df), ('Test', test_df)]):
    ax.hist(np.log1p(split['price_clean']), bins=30, alpha=0.7, color='steelblue')
    ax.set_title(f'{name} (n={len(split)})')
    ax.set_xlabel('log(1 + precio)')
plt.suptitle('Distribución de precios por split (escala log)', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/split_distributions.png', dpi=150)

Split estratificado por deciles de precio — las distribuciones de train/val/test son prácticamente idénticas.

## 1.9 Preview de augmentación

Pipeline: CLAHE → RandomResizedCrop → HorizontalFlip → Rotate → BrightnessContrast → CoarseDropout → Normalize

**Decisión de diseño**: Se desactivó HueSaturationValue porque el color del material (dorado, plateado, rose gold, negro) es una señal de precio importante que no debe distorsionarse.

In [ ]:
sample_img = np.array(Image.open(img_dir / df.iloc[0]['image_name']).convert('RGB'))
transform = get_transforms(config, 'train')

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes[0, 0].imshow(sample_img)
axes[0, 0].set_title('Original')
for ax in axes.flat[1:]:
    aug = transform(image=sample_img)['image']
    aug_np = aug.numpy().transpose(1, 2, 0)
    aug_np = (aug_np * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]).clip(0, 1)
    ax.imshow(aug_np)
for ax in axes.flat:
    ax.axis('off')
plt.suptitle('Augmentation samples (V4: con CLAHE)', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/augmentation_preview.png', dpi=150)

## 1.10 Resumen de decisiones de preprocesamiento

| Decisión | Justificación |
|----------|---------------|
| Log1p en target | Distribución fuertemente sesgada → reduce skewness |
| Resize a 224×224 | Compromiso entre detalle visual y coste computacional |
| CLAHE | Mejora contraste de detalles (esfera, acabados metálicos) |
| Sin HueSaturation | El color del material es señal de precio |
| RandomResizedCrop | Simula zoom parcial al reloj |
| CoarseDropout | Regularización cutout-style |
| Brand embedding | 70 marcas → vector 16-dim aprendido |
| 21 text features | Mecanismo, material, estilo, género del nombre |